# 43 Scene Calibration Comparison

**Purpose:** Compare CLIP scene calibration (class centering, temperature, both) against the configured confidence-weighted branch with grouped cross-validation.

**Inputs:** The labelled development collection (`LABELS_PATH`, 71 videos), the bark detector, CLIP `openai/clip-vit-base-patch32`.

**Outputs:** Tables only; no settings or weights are saved. Section 9 prints optional settings to copy by hand.

**Splits used:** Grouped out-of-fold cross-validation on the development collection (outside the team splits); test: not used (no untouched test set here).

The 71 supplied videos are **development data**; the user authorized their use for fitting and evaluation, and they were previously inspected as test examples. They are not an untouched final test set. The notebook compares the configured confidence-weighted zero-shot branch against unweighted mean logits and centering, temperature, or both. CLIP runs only once per usable bark event. No scene model or calibration artifact is saved or deployed.

Each fold fits centering and temperature on its training groups and predicts held-out groups. One group links any matching dog or original source video, including transitive links. The labeled spans are weak bark labels; they can contain unrelated sounds or changing situations. All methods use the same extracted events, settings, and five category order.

Use the project's Python 3.12 kernel with `pip install -e ".[ml,dev]"`. The existing bark detector requires `third_party/PretrainedSED` and its checkpoint (see notebooks 12-14). The scene model and detector weights must be cached or downloadable. Set `MAX_VIDEOS` for an extraction smoke test, but grouped calibration requires enough independent examples of every class.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score

from woofdecoder.config import REPO_ROOT, load_config
from woofdecoder.preprocess.bark_detector import (
    detect_sed_probs,
    find_dog_events,
    load_bark_detector,
    settings_from_config,
    vocal_score,
)
from woofdecoder.preprocess.media import load_audio_16k, sample_frames
from woofdecoder.preprocess.segments import BarkSegment
from woofdecoder.scene import SceneBranch
from woofdecoder.taxonomy import CATEGORIES
from wooftrainer.scene_trainer.calibration_experiment import (
    connected_groups,
    cross_validate,
    fit_class_bias,
    fit_temperature,
    paired_group_bootstrap,
)

pd.set_option("display.width", 200)

In [ ]:
LABELS_PATH = Path(r"C:/Users/has/Desktop/testing/labels.csv")
VIDEOS_DIR = LABELS_PATH.parent / "videos"
MAX_VIDEOS = None  # e.g. 2 to run only the first two videos
N_FOLDS = 3  # reduce only if training folds cannot cover all classes
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Select data and settings

`labels.csv` must contain source-video URLs, dog IDs, original intervals, and the shared taxonomy labels. All 71 development videos are selected by default. No data is rewritten.

In [ ]:
print("Project:", REPO_ROOT)
print("Categories:", CATEGORIES)

if not LABELS_PATH.is_file() or not VIDEOS_DIR.is_dir():
    raise FileNotFoundError(f"Check LABELS_PATH and VIDEOS_DIR: {LABELS_PATH}, {VIDEOS_DIR}")
if MAX_VIDEOS is not None and (
    isinstance(MAX_VIDEOS, bool) or not isinstance(MAX_VIDEOS, int) or MAX_VIDEOS < 1
):
    raise ValueError("MAX_VIDEOS must be None or a positive integer")
if isinstance(N_FOLDS, bool) or not isinstance(N_FOLDS, int) or N_FOLDS < 2:
    raise ValueError("N_FOLDS must be an integer >= 2")

labels = pd.read_csv(LABELS_PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig")
required = {"video_id", "dog_id", "source_url", "start_s", "end_s", "category"}
if missing := sorted(required - set(labels.columns)):
    raise ValueError(f"Missing labels.csv columns: {missing}")
labels.insert(0, "label_row", np.arange(len(labels)))  # original zero-based data row
for col in ["video_id", "dog_id", "source_url", "category"]:
    labels[col] = labels[col].str.strip()
for col in ["start_s", "end_s"]:
    labels[col] = pd.to_numeric(labels[col], errors="coerce")

valid = (
    labels["video_id"].ne("")
    & labels["dog_id"].ne("")
    & labels["source_url"].ne("")
    & labels["category"].isin(CATEGORIES)
    & np.isfinite(labels["start_s"])
    & np.isfinite(labels["end_s"])
    & labels["start_s"].ge(0)
    & labels["end_s"].gt(labels["start_s"])
)
invalid_labels = labels.loc[~valid].copy()
if not invalid_labels.empty:
    print("Invalid label rows (excluded):")
    display(invalid_labels)
labels = labels.loc[valid].copy()
# Link ALL valid rows before video or event filtering. A row without scored barks
# can still connect two dogs or videos and must not disappear from grouping.
labels["linked_group"] = connected_groups(labels["dog_id"], labels["source_url"])
labels["video_key"] = labels["video_id"].str.casefold()
video_keys = labels["video_key"].drop_duplicates().tolist()
if MAX_VIDEOS is not None:
    video_keys = video_keys[:MAX_VIDEOS]
labels = labels[labels["video_key"].isin(video_keys)].copy()
if labels.empty:
    raise ValueError("No valid labeled rows selected")
video_index = {}
for path in sorted(VIDEOS_DIR.iterdir()):
    if path.is_file() and path.suffix.casefold() == ".mp4":
        video_index.setdefault(path.stem.casefold(), []).append(path)
print(f"Selected {len(labels)} labeled spans from {len(video_keys)} videos; device: {DEVICE}")
print("Annotation counts:")
display(labels.groupby("category").size().reindex(CATEGORIES, fill_value=0).rename("spans"))
print(
    "Duplicate or overlapping spans are retained and treated as ambiguous when an event intersects more than one."
)

## 2. Load models

The existing bark detector runs once per full video. The CLIP branch returns raw, already-scaled logits per frame. The calibration temperatures below are in those same native-logit units; do not apply CLIP's scale twice.

In [ ]:
config = load_config()
# Compare against an uncorrected baseline even if production has a fitted bias.
config["branches"]["scene"]["class_bias"] = [0.0] * len(CATEGORIES)
settings = settings_from_config(config)
print("Bark settings:", settings)
try:
    detector = load_bark_detector(config["bark_detector"]["model"], DEVICE, verbose=False)
except Exception as exc:
    raise RuntimeError(
        "Could not load the bark detector. Check PretrainedSED source, checkpoint and ML "
        "dependencies; see notebooks 12-14 and the exception above."
    ) from exc
try:
    scene = SceneBranch.from_config(config, DEVICE)
except Exception as exc:
    raise RuntimeError(
        "Could not load CLIP. Check the configured model and cached weights."
    ) from exc
print("Loaded CLIP model:", scene.encoder.schema["model"])
print("Prompt order:", scene.encoder.schema["categories"])
if scene.encoder.aggregation != "confidence_weighted_logits":
    raise ValueError(
        "This notebook's current_weighted baseline expects branches.scene.aggregation "
        "to be confidence_weighted_logits. Update the comparison before proceeding."
    )
print(
    "Configured baseline:",
    scene.encoder.aggregation,
    "temperature:",
    scene.encoder.temperature,
    "confidence weight:",
    scene.encoder.confidence_weight,
)

## 3. Extract labeled events

Score events fully contained in exactly one labeled span; report boundary-crossing, ambiguous, missing and outside-span events rather than silently assigning labels. CLIP runs once per matched event and its frame logits are retained in memory.

In [ ]:
def match_event(start, end, spans):
    """Return status and label row; any second overlap makes the event ambiguous."""
    overlaps = spans[(spans["start_s"] < end) & (spans["end_s"] > start)]
    if overlaps.empty:
        return "outside_labeled_spans", None, []
    ids = overlaps["label_row"].tolist()
    if len(overlaps) != 1:
        return "ambiguous_overlapping_labels", None, ids
    row = overlaps.iloc[0]
    if start < row["start_s"] - 1e-6 or end > row["end_s"] + 1e-6:
        return "crosses_label_boundary", None, ids
    return "matched", row, ids


# Report at both event and annotation levels rather than silently dropping anything.
event_records, video_records, scored = [], [], []
for number, key in enumerate(video_keys, 1):
    spans = labels[labels["video_key"] == key]
    paths = video_index.get(key, [])
    print(f"[{number}/{len(video_keys)}] {spans.iloc[0]['video_id']}")
    video_record = {
        "video_id": spans.iloc[0]["video_id"],
        "video_key": key,
        "spans": len(spans),
        "events": 0,
        "status": "ok",
        "detail": "",
    }
    if len(paths) != 1:
        video_record["status"] = "missing_video" if not paths else "ambiguous_video_filename"
        video_record["detail"] = ", ".join(str(p.name) for p in paths)
        video_records.append(video_record)
        continue
    path = paths[0]
    try:
        audio = load_audio_16k(path, verbose=False)
        if audio is None or audio.size == 0:
            video_record["status"] = "no_audio"
            video_records.append(video_record)
            continue
        sed_probs = detect_sed_probs(detector, audio)
        events = find_dog_events(vocal_score(sed_probs), settings)
        video_record["events"] = len(events)
        if not events:
            video_record["status"] = "no_detected_events"
        for start, end in events:
            status, label, overlap_rows = match_event(start, end, spans)
            entry = {
                "video_id": spans.iloc[0]["video_id"],
                "start_s": float(start),
                "end_s": float(end),
                "status": status,
                "overlap_label_rows": overlap_rows,
                "label_row": int(label["label_row"]) if label is not None else None,
                "dog_id": label["dog_id"] if label is not None else None,
                "source_url": label["source_url"] if label is not None else None,
                "linked_group": int(label["linked_group"]) if label is not None else None,
                "category": label["category"] if label is not None else None,
                "frame_count": 0,
                "detail": "",
            }
            if status != "matched":
                event_records.append(entry)
                continue
            try:
                frames, times = sample_frames(path, start, end, verbose=False)
                entry["frame_count"] = len(frames)
                if not frames:
                    entry["status"] = "missing_video_frames"
                else:
                    segment = BarkSegment(
                        start_s=start, end_s=end, source_id=key, frames=frames, frame_times_s=times
                    )
                    result = scene.encoder.encode_segment(segment)
                    logits = result["frame_logits"]
                    if (
                        logits.shape != (len(frames), len(CATEGORIES))
                        or not np.isfinite(logits).all()
                    ):
                        raise ValueError("Invalid CLIP frame logits")
                    entry["status"] = "scored"
                    scored.append(
                        {
                            **entry,
                            "frame_logits": logits,
                            "frame_times_s": times,
                            "configured_probs": result["probs"],
                        }
                    )
            except Exception as exc:  # noqa: BLE001
                entry["status"] = "frame_or_clip_error"
                entry["detail"] = f"{type(exc).__name__}: {exc}"
            event_records.append(entry)
    except Exception as exc:  # noqa: BLE001
        video_record["status"] = "audio_or_detector_error"
        video_record["detail"] = f"{type(exc).__name__}: {exc}"
    video_records.append(video_record)

videos_report = pd.DataFrame(video_records)
events_report = pd.DataFrame(
    event_records,
    columns=[
        "video_id",
        "start_s",
        "end_s",
        "status",
        "overlap_label_rows",
        "label_row",
        "dog_id",
        "source_url",
        "linked_group",
        "category",
        "frame_count",
        "detail",
    ],
)
print("Video status:")
display(videos_report["status"].value_counts().rename("videos"))
print("Event status:")
display(events_report["status"].value_counts().rename("events"))
print(f"Scored {len(scored)} detected events. Unscored detections and affected videos are below.")
display(videos_report[videos_report["status"] != "ok"])
display(events_report[events_report["status"] != "scored"])

## 4. Coverage

Events in the same span share a weak label. Frame counts are sampled images, not independent examples. Spans without a scored bark remain visible below.

In [ ]:
if events_report.empty:
    matched_counts = pd.DataFrame(columns=["label_row", "scored_events"])
else:
    matched_counts = (
        events_report[events_report["status"] == "scored"]
        .groupby("label_row")
        .size()
        .rename("scored_events")
        .reset_index()
    )
coverage = labels.merge(matched_counts, on="label_row", how="left")
coverage["scored_events"] = coverage["scored_events"].fillna(0).astype(int)
coverage = coverage.merge(
    videos_report[["video_key", "status"]].rename(columns={"status": "video_status"}),
    on="video_key",
    how="left",
)
coverage["has_scored_event"] = coverage["scored_events"] > 0
coverage_by_category = (
    coverage.groupby("category")
    .agg(
        labeled_spans=("label_row", "size"),
        spans_scored=("has_scored_event", "sum"),
        scored_events=("scored_events", "sum"),
        dogs=("dog_id", "nunique"),
    )
    .reindex(CATEGORIES, fill_value=0)
)
display(coverage_by_category)
display(
    coverage[
        [
            "label_row",
            "video_id",
            "dog_id",
            "start_s",
            "end_s",
            "category",
            "video_status",
            "scored_events",
        ]
    ]
)

## 5. Fit and compare with grouped cross-validation

Each method predicts the same held-out events. `current_weighted` uses the configured confidence-weighted zero-shot path; `mean_logits` is its unweighted reference. `bias_only`, `temperature_only`, and `bias_and_temperature` use mean logits and fit their corrections exclusively on each fold's training groups. Bias is centered across classes after giving each labeled span equal total weight; temperature minimizes training-fold negative log likelihood with the same span weights.

Mean centering is an **experimental adjustment**, not proof that the offset is prompt bias: it can also remove real class prevalence or visual signal. Temperature helps calibration only if supported by independent evaluation. A fixed fold-training temperature applied after mean logit pooling does not change the predicted class.

In [ ]:
cv_result = None
if not scored:
    print("No scored bark events. Inspect coverage and errors above; calibration cannot run.")
else:
    try:
        cv_result = cross_validate(
            frame_logits=[event["frame_logits"] for event in scored],
            labels=[event["category"] for event in scored],
            span_ids=[str(event["label_row"]) for event in scored],
            dog_ids=[event["dog_id"] for event in scored],
            source_urls=[event["source_url"] for event in scored],
            linked_group_ids=[str(event["linked_group"]) for event in scored],
            n_splits=N_FOLDS,
            alpha=scene.encoder.confidence_weight,
            baseline_temperature=scene.encoder.temperature,
        )
    except ValueError as exc:
        print(f"Cannot run grouped calibration: {exc}")
        print(
            "Inspect coverage. A valid fold needs enough independent groups and all five training categories."
        )

if cv_result is not None:
    # Reproduce the configured branch's actual output before comparing variants.
    np.testing.assert_allclose(
        cv_result["predictions"]["current_weighted"],
        np.stack([event["configured_probs"] for event in scored]),
        rtol=1e-5,
        atol=1e-6,
    )
    print("All methods returned out-of-fold scores for", len(scored), "events.")
    print("Independent dog/source groups:", len(set(cv_result["group_id"])))
    print("Class order:", CATEGORIES)

## 6. Group and class diagnostics

Check fold coverage and fitted parameters before interpreting aggregate metrics. Small, uneven folds can produce unstable estimates.

Temperature fitting is bounded to exp(-6) through exp(6), approximately 0.0025 to 403.4 in native logit units. An estimate near a bound signals instability; a very high temperature makes scores nearly uniform.

In [ ]:
if cv_result is not None:
    indexed = pd.DataFrame(
        {
            "span_id": [str(e["label_row"]) for e in scored],
            "category": [e["category"] for e in scored],
            "dog_id": [e["dog_id"] for e in scored],
            "group_id": cv_result["group_id"],
            "fold": cv_result["fold_id"],
            "frame_count": [e["frame_count"] for e in scored],
        }
    )
    print("Per-class coverage: events, labeled spans and independent dog/source groups")
    display(
        indexed.groupby("category")
        .agg(
            events=("span_id", "size"),
            spans=("span_id", "nunique"),
            dogs=("dog_id", "nunique"),
            groups=("group_id", "nunique"),
        )
        .reindex(CATEGORIES, fill_value=0)
    )
    print("Requested frames per scored event:")
    display(indexed["frame_count"].describe())
    fold_rows = []
    for fold_number, fold in enumerate(cv_result["folds"]):
        for part, positions in (("fit", fold["fit_indices"]), ("held", fold["held_indices"])):
            subset = indexed.iloc[positions]
            for category in CATEGORIES:
                selection = subset[subset["category"] == category]
                fold_rows.append(
                    {
                        "fold": fold_number,
                        "part": part,
                        "category": category,
                        "events": len(selection),
                        "spans": selection["span_id"].nunique(),
                        "groups": selection["group_id"].nunique(),
                    }
                )
    print("Fit and held-out category coverage by fold:")
    display(pd.DataFrame(fold_rows))
    print("Fold-specific temperature estimates and class centering (raw CLIP logit units):")
    display(
        pd.DataFrame(
            [
                {
                    "fold": i,
                    "temperature_only": fold["temperature_only"],
                    "temperature_after_bias": fold["temperature_bias"],
                    **{
                        f"bias_{name}": value
                        for name, value in zip(CATEGORIES, fold["bias"], strict=True)
                    },
                }
                for i, fold in enumerate(cv_result["folds"])
            ]
        )
    )

## 7. Held-out comparisons

Metrics are out-of-fold over development groups, not a final test. Span-weighting reduces the influence of videos with many detections. Compare `mean_logits` with the three corrected versions to isolate calibration; `current_weighted` changes aggregation as well. Scores are relative and not proven calibrated. Keep a new dog- and source-disjoint test set for final reporting.

In [ ]:
if cv_result is not None:
    y_true = cv_result["labels"]
    span_ids = np.array([str(event["label_row"]) for event in scored])
    _, inverse, counts = np.unique(span_ids, return_inverse=True, return_counts=True)
    span_weights = 1.0 / counts[inverse]  # each labeled span has equal total weight
    rows = []
    for method, probabilities in cv_result["predictions"].items():
        top = probabilities.argmax(axis=1)
        target_p = np.clip(probabilities[np.arange(len(y_true)), y_true], 1e-15, 1)
        loss = -np.log(target_p)
        one_hot = np.eye(len(CATEGORIES))[y_true]
        brier = np.sum((probabilities - one_hot) ** 2, axis=1)
        rows.append(
            {
                "method": method,
                "events": len(y_true),
                "spans": len(counts),
                "event_accuracy": accuracy_score(y_true, top),
                "span_weighted_accuracy": accuracy_score(y_true, top, sample_weight=span_weights),
                "span_weighted_macro_f1": f1_score(
                    y_true,
                    top,
                    labels=list(range(len(CATEGORIES))),
                    average="macro",
                    sample_weight=span_weights,
                    zero_division=0,
                ),
                "span_weighted_log_loss": np.average(loss, weights=span_weights),
                "span_weighted_brier": np.average(brier, weights=span_weights),
            }
        )
    print("Pooled out-of-fold scores. Lower log loss and Brier are better; higher F1 is better.")
    display(pd.DataFrame(rows))
    recall = []
    for method, probabilities in cv_result["predictions"].items():
        top = probabilities.argmax(axis=1)
        for category_index, category in enumerate(CATEGORIES):
            mask = y_true == category_index
            recall.append(
                {
                    "method": method,
                    "category": category,
                    "events": int(mask.sum()),
                    "spans": len(np.unique(span_ids[mask])),
                    "span_weighted_recall": np.average(
                        top[mask] == y_true[mask], weights=span_weights[mask]
                    )
                    if mask.any()
                    else np.nan,
                }
            )
    print("Per-category recall (each span gets equal total weight):")
    display(pd.DataFrame(recall))
    print("These OOF comparisons are exploratory; do not select on a reserved final test set.")
else:
    print("No comparison generated; resolve the coverage or grouping issue above.")

In [ ]:
if cv_result is not None:
    view = pd.DataFrame(
        {
            "video_id": [e["video_id"] for e in scored],
            "start_s": [e["start_s"] for e in scored],
            "end_s": [e["end_s"] for e in scored],
            "category": [e["category"] for e in scored],
            "fold": cv_result["fold_id"],
            "group": cv_result["group_id"],
        }
    )
    for method, probabilities in cv_result["predictions"].items():
        view[method] = [CATEGORIES[i] for i in probabilities.argmax(axis=1)]
    print("Example held-out event predictions; raw score arrays are in cv_result['predictions'].")
    display(view.head(30))

## 8. Uncertainty in paired log-loss differences

Resample whole linked dog/source groups 1,000 times and compute each candidate's span-weighted log-loss minus the baseline's. Negative differences favor the candidate. These approximate intervals are conditional on the already-fitted out-of-fold predictions; they do **not** include uncertainty from refitting models or choosing methods. Wide intervals and few groups should temper conclusions.

In [ ]:
if cv_result is not None:
    comparisons = []
    for baseline in ("current_weighted", "mean_logits"):
        for candidate in ("bias_only", "temperature_only", "bias_and_temperature"):
            interval = paired_group_bootstrap(
                cv_result["predictions"][candidate],
                cv_result["predictions"][baseline],
                cv_result["labels"],
                [str(event["label_row"]) for event in scored],
                cv_result["group_id"],
                n_boot=1000,
                seed=42,
            )
            comparisons.append({"candidate": candidate, "baseline": baseline, **interval})
    print("Paired grouped bootstrap: negative log-loss difference favors the candidate.")
    display(pd.DataFrame(comparisons))
else:
    print("No held-out predictions to compare.")

## 9. Optional: fit settings for new, independent videos

Run the next cell only after extracting and checking all development events above. It fits the same unweighted mean-logit bias and temperature tested here, on all usable development spans, and prints settings to copy into the scene configuration. The fit does not write files or change the active configuration. The temperature can be unstable with few groups.

**Do not use these full-development fitted scores to train fusion on the same 71 videos:** use the out-of-fold scene predictions above instead. Evaluate the complete pipeline on separate dogs and source videos.

In [ ]:
# Optional: fit for inference on new, independent videos, not fusion training on these 71.

if MAX_VIDEOS is not None:
    raise ValueError("Set MAX_VIDEOS to None and rerun extraction before a full-development fit.")
if not scored:
    raise ValueError("No scored events: complete and inspect extraction before fitting.")
# This fit includes only scored events: review the coverage and error reports above.

fit_logits = np.stack([event["frame_logits"].mean(axis=0) for event in scored])
fit_labels = np.array([CATEGORIES.index(event["category"]) for event in scored])
fit_spans = np.array([str(event["label_row"]) for event in scored])
fit_bias = fit_class_bias(fit_logits, fit_labels, fit_spans)
fit_t = fit_temperature(fit_logits - fit_bias, fit_labels, fit_spans)

print("Fitted on all usable development spans; use only on independent videos.")
print("Class order:", CATEGORIES)
print("Model:", scene.encoder.schema["model"], "revision:", scene.encoder.schema["revision"])
print("Prompts used for this fit (must match inference):", scene.encoder.schema["prompts"])
print("branches:\n  scene:")
print("    aggregation: mean_frame_logits  # matches this fitted calibration")
print("    class_bias: [" + ", ".join(f"{value:.10g}" for value in fit_bias) + "]")
print(f"    temperature: {fit_t:.10g}")

## Results and decisions

No settings or weights are saved by this notebook. If a correction is later used to train fusion on these same examples, supply its **out-of-fold** scene scores to fusion and evaluate the entire pipeline on separate dogs and sources.